# Vesuvius Labelled Segment Visualizer

Explore 33-layer PHercParis4 segments from `data/labelled_segments` with their pseudo-labels.

Labels are **continuous probability maps** (0–255) produced by a `tile64-stride16` ink-detection model — not manual annotations.

**Sections:** label overview · interactive layer browser · MIP vs label comparison · orthogonal projections · label statistics

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import tifffile as tiff
import ipywidgets as widgets
from IPython.display import display

DATA_DIR = Path("../data/labelled_segments")
SEGMENTS = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])

def load_slice(seg_dir, z):
    return tiff.imread(str(seg_dir / "surface_volume" / f"{z:02d}.tif")).astype(np.float32)

def normalize(arr, mask=None):
    src = arr[mask] if mask is not None else arr.ravel()
    p1, p99 = np.percentile(src, [1, 99])
    return np.clip((arr - p1) / (p99 - p1 + 1e-8), 0, 1)

seg_sel = widgets.Dropdown(options=SEGMENTS, description="Segment:")
print(f"Found {len(SEGMENTS)} labelled segments:")
for s in SEGMENTS:
    n = len(list((DATA_DIR / s / "surface_volume").glob("*.tif")))
    sz = tiff.imread(str(DATA_DIR / s / "ink_labels.tif")).shape
    print(f"  {s}  ({n} layers, label {sz})")
display(seg_sel)

## 1. Load Data

Re-run this cell after changing the dropdown to switch segments.

In [ ]:
SEG_ID  = seg_sel.value
seg_dir = DATA_DIR / SEG_ID
tifs     = sorted((seg_dir / "surface_volume").glob("*.tif"))
num_layers = len(tifs)

# Labels: uint8 probability map 0-255 from pseudo-labeller
labels_raw  = tiff.imread(str(seg_dir / "ink_labels.tif")).astype(np.float32)
labels_prob = labels_raw / 255.0 if labels_raw.max() > 1.0 else labels_raw
labels_bin  = (labels_prob >= 0.5).astype(np.uint8)   # threshold at 0.5

# Mask: not all labelled segments have one — use full frame as fallback
mask_path = seg_dir / "mask.png"
mask = np.array(Image.open(str(mask_path)).convert("L")) > 128 if mask_path.exists() else np.ones(labels_raw.shape, dtype=bool)

print(f"Segment     : {SEG_ID}")
print(f"Layers      : {num_layers}")
print(f"Label shape : {labels_raw.shape}  dtype={labels_raw.dtype}  range=[{labels_raw.min():.0f}, {labels_raw.max():.0f}]")
print(f"Mask source : {'mask.png' if mask_path.exists() else 'all-ones fallback'}")
ink_pct = labels_bin.sum() / labels_bin.size * 100
print(f"Ink coverage: {ink_pct:.1f}%  (label ≥ 0.5 threshold)")

## 2. Label Overview

The probability map encodes pseudo-label confidence. The ignore band (0.4–0.6) is excluded during training.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im0 = axes[0].imshow(labels_prob, cmap="hot", vmin=0, vmax=1)
plt.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
axes[0].set_title("Pseudo-label probability map"); axes[0].axis("off")

axes[1].imshow(labels_bin, cmap="gray")
axes[1].set_title("Binarized labels  (threshold = 0.5)"); axes[1].axis("off")

non_zero = labels_prob[labels_prob > 0.01].ravel()
axes[2].hist(non_zero, bins=100, color="tomato", edgecolor="none", density=True)
axes[2].axvline(0.5, color="k",  ls="--", lw=1.5, label="threshold = 0.5")
axes[2].axvspan(0.4, 0.6, alpha=0.15, color="gray", label="ignore band (0.4–0.6)")
axes[2].set_title("Label probability distribution (non-zero)")
axes[2].set_xlabel("Probability"); axes[2].set_ylabel("Density")
axes[2].legend()

plt.suptitle(f"{SEG_ID} — label overview", fontsize=13)
plt.tight_layout(); plt.show()

## 3. Interactive Layer Browser

Scrub through all Z-layers. Red tint in the overlay = ink probability from pseudo-label.

In [ ]:
@widgets.interact(z=widgets.IntSlider(min=0, max=num_layers-1, value=num_layers//2, description="Layer"))
def view_layer(z=num_layers//2):
    sl   = load_slice(seg_dir, z)
    disp = normalize(sl, mask)

    overlay = np.stack([disp, disp, disp], axis=-1)
    overlay[:, :, 0] = np.clip(overlay[:, :, 0] + labels_prob * 0.65, 0, 1)
    overlay[:, :, 1] = np.clip(overlay[:, :, 1] - labels_prob * 0.30, 0, 1)
    overlay[:, :, 2] = np.clip(overlay[:, :, 2] - labels_prob * 0.30, 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"{SEG_ID} — Layer z={z:02d}", fontsize=12)

    axes[0].imshow(disp, cmap="gray")         ; axes[0].set_title("CT slice")                        ; axes[0].axis("off")
    axes[1].imshow(np.clip(overlay, 0, 1))    ; axes[1].set_title("CT + pseudo-label overlay (red=ink)") ; axes[1].axis("off")
    axes[2].imshow(labels_prob, cmap="hot", vmin=0, vmax=1) ; axes[2].set_title("Label probability map") ; axes[2].axis("off")

    plt.tight_layout(); plt.show()

## 4. MIP vs Label Comparison

The Maximum Intensity Projection collapses all Z-layers into one image — useful for comparing spatial ink distribution against the label.

In [ ]:
DOWNSAMPLE = 4
print(f"Loading volume (spatial ×{DOWNSAMPLE} downsample)...")
vol = np.stack(
    [load_slice(seg_dir, z)[::DOWNSAMPLE, ::DOWNSAMPLE] for z in range(num_layers)],
    axis=0
)
print(f"Volume shape: {vol.shape}  ({vol.nbytes/1e6:.0f} MB)")

mip        = vol.max(axis=0)
labels_ds  = labels_prob[::DOWNSAMPLE, ::DOWNSAMPLE]

fig, axes = plt.subplots(1, 4, figsize=(22, 6))

axes[0].imshow(normalize(mip), cmap="gray")
axes[0].set_title("CT MIP (max across Z)"); axes[0].axis("off")

im1 = axes[1].imshow(labels_ds, cmap="hot", vmin=0, vmax=1)
plt.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
axes[1].set_title("Pseudo-label probability"); axes[1].axis("off")

axes[2].imshow(labels_ds >= 0.5, cmap="gray")
axes[2].set_title("Binarized label"); axes[2].axis("off")

# MIP + label overlay
overlay = np.stack([normalize(mip)] * 3, axis=-1)
overlay[:, :, 0] = np.clip(overlay[:, :, 0] + labels_ds * 0.7, 0, 1)
overlay[:, :, 1] = np.clip(overlay[:, :, 1] - labels_ds * 0.3, 0, 1)
overlay[:, :, 2] = np.clip(overlay[:, :, 2] - labels_ds * 0.3, 0, 1)
axes[3].imshow(np.clip(overlay, 0, 1))
axes[3].set_title("MIP + label overlay (red=ink)"); axes[3].axis("off")

plt.suptitle(f"{SEG_ID} — MIP vs pseudo-labels", fontsize=13)
plt.tight_layout(); plt.show()

## 5. Orthogonal Projections

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
views = [
    ("MIP — axial (Z→)",    vol.max(0)),
    ("MIP — coronal (Y→)",  vol.max(1)),
    ("MIP — sagittal (X→)", vol.max(2)),
    ("Mean — axial",         vol.mean(0)),
    ("Mean — coronal",        vol.mean(1)),
    ("Mean — sagittal",       vol.mean(2)),
]
for ax, (title, img) in zip(axes.ravel(), views):
    ax.imshow(img, cmap="gray"); ax.set_title(title); ax.axis("off")

plt.suptitle(f"{SEG_ID} — orthogonal projections (downsample={DOWNSAMPLE})", fontsize=13)
plt.tight_layout(); plt.show()

## 6. Label Statistics

Spatial distribution and coverage breakdown.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Full histogram (log scale for tail visibility)
axes[0].hist(labels_prob.ravel(), bins=100, color="tomato", edgecolor="none", log=True)
axes[0].axvline(0.5, color="k", ls="--", lw=1, label="threshold")
axes[0].axvspan(0.4, 0.6, alpha=0.12, color="gray", label="ignore band")
axes[0].set_title("Full label distribution (log scale)")
axes[0].set_xlabel("Probability"); axes[0].legend()

# Row-wise ink density
row_ink = (labels_prob >= 0.5).mean(axis=1)
axes[1].plot(row_ink, np.arange(len(row_ink)), lw=1)
axes[1].invert_yaxis()
axes[1].set_title("Row-wise ink density")
axes[1].set_xlabel("Fraction of pixels ≥ 0.5"); axes[1].set_ylabel("Row (Y)")
axes[1].grid(True, ls="--", alpha=0.4)

# Column-wise ink density
col_ink = (labels_prob >= 0.5).mean(axis=0)
axes[2].plot(np.arange(len(col_ink)), col_ink, lw=1)
axes[2].set_title("Column-wise ink density")
axes[2].set_xlabel("Column (X)"); axes[2].set_ylabel("Fraction of pixels ≥ 0.5")
axes[2].grid(True, ls="--", alpha=0.4)

plt.suptitle(f"{SEG_ID} — spatial label statistics", fontsize=12)
plt.tight_layout(); plt.show()

# Summary table
thresholds = [0.1, 0.3, 0.5, 0.7, 0.9]
print(f"{'Threshold':<12} {'Ink pixels':>12} {'Coverage':>10}")
print("-" * 36)
for t in thresholds:
    n = (labels_prob >= t).sum()
    pct = n / labels_prob.size * 100
    print(f"{t:<12.1f} {n:>12,} {pct:>9.2f}%")